`RecursiveCharacterTextSplitter` 是 LangChain 中**最常用、也是官方推荐默认使用**的文本分割器。

它的核心设计理念是：**尽可能保持语义的完整性**。

不同于简单的按字符数硬切（比如每 100 个字切一刀），它是“智能”的：它会优先在段落之间切分，如果段落太长，再在句子之间切分，实在不行才会在单词之间切分。

---

### 1. 核心原理：为什么叫 "Recursive" (递归)？

它之所以叫“递归”，是因为它通过**递归地尝试**一系列分隔符列表来分割文本。

默认的分隔符顺序如下（优先级从高到低）：

1. `"\n\n"` (双换行符)：首先尝试在段落之间切分。
2. `"\n"` (单换行符)：如果段落超过了 `chunk_size`，就尝试在行之间切分。
3. `" "` (空格)：如果行还太长，就在单词之间切分。
4. `""` (空字符)：最后手段，强制截断字符。

**目的：** 尽量让相关的文本块（chunk）保持在一起（例如，同一段落的句子通常在语义上是相关的）。

---

### 2. 基础代码示例

#### 初始化参数

在使用前，理解这两个参数至关重要：

* **`chunk_size`**: 每个片段的**目标**大小（默认单位是字符数）。注意这是一个软限制，分割器会尽量向这个数值靠拢。
* **`chunk_overlap`**: 片段之间的**重叠部分**。这能防止上下文在切分点丢失（比如一个问题的答案正好跨越了切分点）。

#### 示例代码

---

**输出预览（逻辑演示）：**
你会发现切分点通常很自然，不会在句子中间突然断开（除非句子本身就超过了 50 个字符）。

---

### 3. 常见应用场景

#### A. RAG 系统 (Retrieval-Augmented Generation)

这是最典型的用法。

* **问题**：如果不切分，直接把整本书塞给向量数据库，检索时召回的粒度太粗，且容易超过 LLM 的 Context Window。
* **解决**：使用 `RecursiveCharacterTextSplitter` 将文档切分为 500-1000 字符的片段。
* **优势**：保证检索出来的片段是语义完整的段落，而不是断章取义的半句话。

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 准备一段长文本（模拟一篇技术文章）
long_text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架。
它使应用程序能够：
1. 具有上下文感知能力：将语言模型连接到上下文来源（提示指令，少量的示例，内容等）。
2. 进行推理：依靠语言模型进行推理（根据提供的上下文如何回答，采取什么行动等）。

主要价值在于：
LangChain 提供了标准化的内存接口。
LangChain 提供了多种 Model IO 的支持。
"""

# 2. 初始化分割器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,  # 目标每个块 50 个字符
    chunk_overlap=10,  # 块之间重叠 10 个字符
    add_start_index=True,  # 在元数据中记录切分位置
)

# 3. 执行切分
# create_documents 会自动把文本封装成 Document 对象
docs = text_splitter.create_documents([long_text])

# 4. 打印结果查看
for i, doc in enumerate(docs):
    print(f"--- Chunk {i} ({len(doc.page_content)} chars) ---")
    print(doc.page_content)

--- Chunk 0 (43 chars) ---
LangChain 是一个用于开发由语言模型驱动的应用程序的框架。
它使应用程序能够：
--- Chunk 1 (43 chars) ---
1. 具有上下文感知能力：将语言模型连接到上下文来源（提示指令，少量的示例，内容等）。
--- Chunk 2 (41 chars) ---
2. 进行推理：依靠语言模型进行推理（根据提供的上下文如何回答，采取什么行动等）。
--- Chunk 3 (30 chars) ---
主要价值在于：
LangChain 提供了标准化的内存接口。
--- Chunk 4 (29 chars) ---
LangChain 提供了多种 Model IO 的支持。


**输出预览（逻辑演示）：**
你会发现切分点通常很自然，不会在句子中间突然断开（除非句子本身就超过了 50 个字符）。

---

### 3. 常见应用场景

#### A. RAG 系统 (Retrieval-Augmented Generation)

这是最典型的用法。

* **问题**：如果不切分，直接把整本书塞给向量数据库，检索时召回的粒度太粗，且容易超过 LLM 的 Context Window。
* **解决**：使用 `RecursiveCharacterTextSplitter` 将文档切分为 500-1000 字符的片段。
* **优势**：保证检索出来的片段是语义完整的段落，而不是断章取义的半句话。

#### B. 代码切分 (Code Splitting)

LangChain 专门为代码优化了递归逻辑。它不是按 `\n\n` 切，而是按编程语言的语法结构切（比如 Python 的 `def`, `class`）。

In [3]:
from langchain_text_splitters import Language


python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=100, chunk_overlap=0
)

python_code = """
def hello_world():
    print("Hello")

class AI:
    def __init__(self):
        pass
"""

docs = python_splitter.create_documents([python_code])

### 4. 关键参数调优建议

| 参数 | 建议值 | 说明 |
| --- | --- | --- |
| `chunk_size` | **500 - 1000** | 这个范围通常最适合大多数 LLM (如 GPT-3.5/4) 的理解能力。太小会导致语义碎片化，太大则会导致检索不精准。 |
| `chunk_overlap` | **10% - 20%** | 例如 chunk_size=500，overlap 设为 50-100。这能确保句子之间的连贯性。 |
| `separators` | **默认即可** | 除非你在处理非常特殊的格式（如 markdown 表格或特定日志格式），否则不需要手动修改分隔符列表。 |
| `length_function` | `len` | 默认按**字符数**计算长度。如果你对 Token 限制非常严格，可以换成 `tiktoken_len`（按 Token 计数），但计算速度会变慢。 |

### 总结

`RecursiveCharacterTextSplitter` 是处理自然语言文档的**首选**。它用“递归降级”的策略，在满足长度限制的前提下，最大程度地保留了文本的可读性和逻辑结构。

**我能为你做的下一步：**
既然你对 Python 感兴趣，需要我展示 **`RecursiveCharacterTextSplitter.from_language(Language.PYTHON)`** 的具体效果吗？它可以精准地按照 Python 的类和函数边界来切分代码，非常适合做代码库的问答助手。